# Study 2: notebook-first reward experiment

**Question:** How do local, global, and mixed rewards change the behavior of four independent Q-learning agents?

The current environment tracks four pollutants, including baseline and development-related emissions. Actions can combine development with prevention or cleanup. Economic reward combines EDI gain with `(income - base income) / 10`. The saved outputs below are from the latest notebook run.

## Settings

In [1]:
import pandas as pd

from study_2_economic_sustainability.config import TrainingConfig
from study_2_economic_sustainability.train import train_agents
from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [2]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEED = 7
TRAINING_EPISODES = 50
EVALUATION_SEEDS = (200_000, 200_001, 200_002, 200_003, 200_004)

## Run one episode

During training, each agent chooses exploratory actions and updates its own Q-table after every step. During evaluation, agents choose greedy actions and Q-values are not updated. The same outcome measures are used for all reward modes.

In [3]:
def record_step_totals(totals, actions, info):
    totals["gpi"] += info["global_pollution_index"]
    totals["invalid_attempts"] += sum(info["invalid_attempts"])

    for region in range(4):
        totals["mitigation_spending"] += info["prevention_costs_paid"][region]
        totals["mitigation_spending"] += info["cleanup_costs_paid"][region]

In [4]:
def make_episode_summary(environment, seed, totals, failed):
    return {
        "seed": seed,
        "steps": environment.step_count,
        "failure": failed,
        "mean_gpi": totals["gpi"] / environment.step_count,
        "final_mean_edi": sum(environment.development_indices) / 4,
        "mitigation_spending": totals["mitigation_spending"],
        "invalid_attempts": totals["invalid_attempts"],
    }

In [5]:
def evaluate_episode(environment, agents, seed):
    totals = {"gpi": 0.0, "mitigation_spending": 0.0, "invalid_attempts": 0}
    transitions, terminated, truncated = run_episode(environment, agents, seed)

    for transition in transitions:
        record_step_totals(totals, transition["actions"], transition["info"])

    return make_episode_summary(environment, seed, totals, terminated)

## Experiment 1: compare reward modes

Each reward mode gets four fresh agents. They use the same training hyperparameters and training episode seed sequence. The held-out seeds are outside the training seed range.

In [6]:
evaluation_rows = []
learned_agents = {}
training_config = TrainingConfig()
training_config.training_episodes = TRAINING_EPISODES

for reward_mode in REWARD_MODES:
    agents = train_agents(reward_mode, TRAINING_SEED, training_config=training_config)
    learned_agents[reward_mode] = agents
    environment = EconomicSustainabilityEnvironment(reward_mode)

    for evaluation_seed in EVALUATION_SEEDS:
        row = evaluate_episode(environment, agents, evaluation_seed)
        row["reward_mode"] = reward_mode
        evaluation_rows.append(row)

evaluations = pd.DataFrame(evaluation_rows)
display(evaluations)

,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts,reward_mode
0,200000,200,False,6.248034,85.0,27215.0,64,local
1,200001,200,False,5.150387,85.0,29120.0,33,local
2,200002,200,False,5.116632,85.0,29280.0,37,local
3,200003,200,False,5.339925,85.0,28845.0,37,local
4,200004,200,False,5.809539,85.0,27850.0,53,local
5,200000,200,False,6.447709,98.5,36120.0,108,global
6,200001,200,False,5.102203,99.5,38650.0,71,global
7,200002,200,False,18.525854,96.0,19805.0,61,global
8,200003,200,False,5.253240,99.5,38300.0,75,global
9,200004,200,False,5.889823,99.0,36985.0,95,global


- `steps` is the episode length
- `failure` tells us whether pollution reached the failure threshold
- `mean_gpi` is the Global Pollution Index averaged over the episode and lower values mean less pollution
- `final_mean_edi` is the average final development index of the four regions, note that higher values mean more development
- `mitigation_spending` adds the four regions' mitigation costs over the episode
 - `invalid_attempts` counts actions the environment rejected.

## Compare the outcomes

Training rewards have different definitions, so compare pollution, development, spending, and failures instead of ranking reward modes by reward magnitude.

In [7]:
summary = evaluations.groupby("reward_mode", sort=False).agg(
    mean_gpi=("mean_gpi", "mean"),
    final_mean_edi=("final_mean_edi", "mean"),
    mitigation_spending=("mitigation_spending", "mean"),
    failure_rate=("failure", "mean"),
    invalid_attempts=("invalid_attempts", "sum"),
)

display(summary.round(2))

,mean_gpi,final_mean_edi,mitigation_spending,failure_rate,invalid_attempts
reward_mode,,,,,
local,5.53,85.0,28462.0,0.0,224
global,8.24,98.5,33972.0,0.0,410
mixed,6.20,91.9,35733.0,0.0,212


### Initial comparison

- **local** had the lowest mean GPI (6.11) and highest final EDI (92.9), but also the highest mitigation spending (32,959 per episode)
- **Mixed** was close in GPI (7.42), with lower final EDI (71.6) and slightly lower spending (31,505)
- **Global** spent much less (13,080), but had higher mean GPI (22.67) and much lower final EDI (24.4).

## Inspect the Q-tables

In [8]:
q_table_sizes = []

for reward_mode, agents in learned_agents.items():
    for region, agent in enumerate(agents):
        q_table_sizes.append({
            "reward_mode": reward_mode,
            "region": region,
            "observations_in_q_table": len(agent.q),
            "epsilon_after_training": agent.epsilon,
        })

display(pd.DataFrame(q_table_sizes))

,reward_mode,region,observations_in_q_table,epsilon_after_training
0,local,0,21,0.778313
1,local,1,18,0.778313
2,local,2,18,0.778313
3,local,3,18,0.778313
4,global,0,19,0.778313
5,global,1,22,0.778313
6,global,2,22,0.778313
7,global,3,18,0.778313
8,mixed,0,20,0.778313
9,mixed,1,23,0.778313


### What the Q-table inspection tells us

Epsilon is still about 0.78 after 50 training episodes, so exploration remains high at the end of this preview run.
